In [13]:
import math 

x = [1,2]
y = [1,0] 

math.sqrt((x[0] - y[0])**2 + (x[1] - y[1])**2 )

2.0

In [ ]:
(x[0]*y[0] + x[1]*y[1])/math.sqrt(2) # tendenziell Skalarprodukt kleiner als euklidische distanz 

0.7071067811865475

In [15]:
import torch 

In [16]:
sentence = "this is a sample sentence for embedding"
sentence2 = "this is sentence embedding"

dc = {s:i for i,s in enumerate(sorted(sentence.replace(',', '').split()))}
dc

{'a': 0,
 'embedding': 1,
 'for': 2,
 'is': 3,
 'sample': 4,
 'sentence': 5,
 'this': 6}

In [17]:
sentence_int = torch.tensor( [dc[s] for s in sentence.replace(',', '').split()])
print(sentence_int) # Tokenized sequence

tensor([6, 3, 0, 4, 5, 2, 1])


In [ ]:
emb = torch.nn.Embedding(len(dc), 3) # 3... Dimension, in Praxis ist die Dimension 384 - 4096
emb.weight.data # alles lernbare Parameter 


tensor([[-0.0876,  0.8562, -1.0949],
        [ 0.4410,  0.4499, -0.5287],
        [ 0.3639, -0.4322,  2.1575],
        [-0.7833, -0.7064, -0.8430],
        [-0.7571, -0.5136, -0.2402],
        [ 0.8982, -2.7058,  0.8192],
        [-0.7997, -0.3182, -1.3056]])

In [ ]:
sentence_emb = emb(sentence_int).detach() # .detach() = aus autograd rausnehmen
sentence_emb

tensor([[-0.7997, -0.3182, -1.3056],
        [-0.7833, -0.7064, -0.8430],
        [-0.0876,  0.8562, -1.0949],
        [-0.7571, -0.5136, -0.2402],
        [ 0.8982, -2.7058,  0.8192],
        [ 0.3639, -0.4322,  2.1575],
        [ 0.4410,  0.4499, -0.5287]])

In [ ]:
context_length = 128 
pos_emb = torch.nn.Embedding(128,3)
pos_emb.weight.data

In [30]:
pos_list = list(range(len(dc)))
sentence_pos = pos_emb(torch.tensor(pos_list)).detach()

In [33]:
# Input für Transformer

sentence_emb + sentence_pos

tensor([[-0.5178,  0.3587, -3.0478],
        [-2.3411, -0.9986, -0.5647],
        [ 0.0738,  1.3753, -0.4078],
        [-1.7932, -1.5886,  1.4647],
        [ 0.7118, -3.3622,  1.6253],
        [ 0.1986, -1.8750,  3.1510],
        [ 0.1233,  0.6720,  0.7639]])

In [39]:
# Q, K, V

d = sentence_emb.shape[1] # Dimension von Embedding 
d_Q, d_K, d_V = 2, 2, 4 # Dimensionen für Q, K , V

W_Q_L = torch.nn.Linear(d_Q, d, bias=False) # Matrix W_Q 
# oder
W_Q = torch.nn.Parameter(torch.randn(d,d_Q))
W_K = torch.nn.Parameter(torch.randn(d,d_K))
W_V = torch.nn.Parameter(torch.randn(d,d_V))


In [45]:
queries = sentence_emb @ W_Q # Alle Queries
keys = sentence_emb @ W_K # Alle Keys
values = sentence_emb @ W_V # Alle Values

scores_raw = queries @ keys.T
scores_raw

tensor([[  2.2609,   2.1608,   0.3445,   1.5297,   0.1227,  -1.8151,  -0.4868],
        [  1.0369,   0.7410,   0.7198,   0.5110,  -1.5864,  -1.2660,   0.0202],
        [  2.8534,   3.2885,  -0.8274,   2.3585,   3.8453,  -1.3167,  -1.1613],
        [  0.2521,   0.0423,   0.4850,   0.0191,  -1.2920,  -0.5470,   0.1392],
        [ -5.0377,  -6.4141,   2.8281,  -4.6276, -10.7866,   1.2696,   2.6426],
        [ -4.2767,  -4.5308,   0.3452,  -3.2315,  -3.1468,   2.6642,   1.3527],
        [  1.1306,   1.3340,  -0.3977,   0.9582,   1.7279,  -0.4678,  -0.4904]],
       grad_fn=<MmBackward0>)

In [47]:
import torch.nn.functional as F 
torch.set_printoptions(sci_mode=False)
scores = scores_raw/math.sqrt(d_K) # Skalierungsschritt 
attn_weights = F.softmax(scores, dim=1) # Softmax über Zeilen
attn_weights


tensor([[0.3120, 0.2906, 0.0805, 0.1860, 0.0688, 0.0175, 0.0447],
        [0.2416, 0.1960, 0.1930, 0.1665, 0.0378, 0.0474, 0.1177],
        [0.1899, 0.2583, 0.0141, 0.1338, 0.3829, 0.0100, 0.0111],
        [0.1749, 0.1508, 0.2062, 0.1484, 0.0587, 0.0994, 0.1615],
        [0.0017, 0.0007, 0.4505, 0.0023, 0.0000, 0.1497, 0.3951],
        [0.0045, 0.0038, 0.1187, 0.0095, 0.0100, 0.6116, 0.2420],
        [0.1803, 0.2082, 0.0612, 0.1596, 0.2751, 0.0582, 0.0573]],
       grad_fn=<SoftmaxBackward0>)

In [48]:
attn_output = attn_weights @ values
attn_output # Kontextvector

tensor([[ 0.4413,  1.6974, -0.6126,  0.2927],
        [ 0.0702,  1.4910, -0.3252, -0.0197],
        [ 1.7860, -0.0145, -0.8691,  1.4452],
        [ 0.1279,  0.9306, -0.2282,  0.0502],
        [-0.7228,  0.4523,  0.4908, -0.6863],
        [ 0.4016, -2.0687,  0.0392,  0.4894],
        [ 1.2825,  0.1751, -0.6757,  1.0337]], grad_fn=<MmBackward0>)

In [52]:
block_size = attn_weights.shape[0]
mask = torch.tril(torch.ones(block_size, block_size))
attn_weights * mask 

tensor([[0.3120, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2416, 0.1960, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1899, 0.2583, 0.0141, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1749, 0.1508, 0.2062, 0.1484, 0.0000, 0.0000, 0.0000],
        [0.0017, 0.0007, 0.4505, 0.0023, 0.0000, 0.0000, 0.0000],
        [0.0045, 0.0038, 0.1187, 0.0095, 0.0100, 0.6116, 0.0000],
        [0.1803, 0.2082, 0.0612, 0.1596, 0.2751, 0.0582, 0.0573]],
       grad_fn=<MulBackward0>)

In [56]:
mask = torch.tril(torch.ones(block_size, block_size), diagonal=-1)
scores_masked = scores.masked_fill(mask.T.bool(), -torch.inf)
attn_weights_masked = torch.softmax(scores_masked, dim=1)
attn_weights_masked

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5521, 0.4479, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4108, 0.5588, 0.0304, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2571, 0.2217, 0.3031, 0.2181, 0.0000, 0.0000, 0.0000],
        [0.0038, 0.0014, 0.9896, 0.0051, 0.0001, 0.0000, 0.0000],
        [0.0060, 0.0050, 0.1565, 0.0125, 0.0133, 0.8068, 0.0000],
        [0.1803, 0.2082, 0.0612, 0.1596, 0.2751, 0.0582, 0.0573]],
       grad_fn=<SoftmaxBackward0>)